# FD004 Paper 2 — Shift-V2 / DS-B with frozen P2-E8 specification

## Objective

Evaluate the selected **P2-E8 60-cycle point predictor** under the frozen Paper 1 **DS-B restricted operating-envelope shift**.

### What remains frozen from Paper 1 DS-B
- same engine-level train / calibration / test separation
- all six regimes remain represented
- regime-specific central operating envelope estimated from training engines only
- train/calibration endpoints are inside the envelope
- test endpoints are outside the envelope
- sensor scaling is fit only on training-engine, inside-envelope observations
- complete consecutive trajectory history is preserved

### P2-E8 substitution
- 60-cycle temporal context
- 20D input → Dense(18, no bias) → LSTM(64) → Dense(1)
- Adam 1e-3 / MSE
- maximum 60 epochs, batch size 64, patience 10

This notebook performs **point-model Shift-V2 validation only**. Conformal uncertainty is deferred to UQ-V1.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Imports and constants

In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

DATA_DIR = "/content/drive/MyDrive/CMAPSSData"
OUTPUT_DIR = "/content/drive/MyDrive/FD004_Paper2_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RUL_CLIP = 125
WINDOW_SIZE = 60
N_REGIMES = 6
ENVELOPE_QUANTILE = 0.80

OPERATING_SETTINGS = [
    "setting_1",
    "setting_2",
    "setting_3"
]

SELECTED_SENSORS = [
    "s_2", "s_3", "s_4", "s_7", "s_8", "s_9",
    "s_11", "s_12", "s_13", "s_14",
    "s_15", "s_17", "s_20", "s_21"
]

ALL_RAW_COLS = (
    ["unit_id", "time_cycles"]
    + OPERATING_SETTINGS
    + [f"s_{i}" for i in range(1, 22)]
)

print("DS-B configuration loaded.")
print("Envelope quantile:", ENVELOPE_QUANTILE)
print("Output directory:", OUTPUT_DIR)

DS-B configuration loaded.
Envelope quantile: 0.8
Output directory: /content/drive/MyDrive/FD004_Paper2_Results


## 2. Load raw FD004 training data

In [3]:
train_path = os.path.join(DATA_DIR, "train_FD004.txt")

train_raw = pd.read_csv(
    train_path,
    sep=r"\s+",
    header=None
)

train_raw = train_raw.iloc[:, :len(ALL_RAW_COLS)]
train_raw.columns = ALL_RAW_COLS

print("FD004 training data loaded.")
print("Shape:", train_raw.shape)
print("Engines:", train_raw["unit_id"].nunique())

display(train_raw.head())

FD004 training data loaded.
Shape: (61249, 26)
Engines: 249


,unit_id,time_cycles,setting_1,setting_2,setting_3,s_1,s_2,s_3,s_4,s_5,...,s_12,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,129.78,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,312.59,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,129.62,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,129.80,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,164.11,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754


## 3. Calculate clipped RUL

In [4]:
max_cycles = (
    train_raw.groupby("unit_id")["time_cycles"]
    .max()
    .rename("max_cycle")
)

train_raw = train_raw.merge(
    max_cycles,
    on="unit_id",
    how="left"
)

train_raw["RUL_raw"] = (
    train_raw["max_cycle"] - train_raw["time_cycles"]
)

train_raw["RUL"] = train_raw["RUL_raw"].clip(
    upper=RUL_CLIP
)

display(
    train_raw[
        ["unit_id", "time_cycles", "RUL_raw", "RUL"]
    ].head(10)
)

,unit_id,time_cycles,RUL_raw,RUL
0,1,1,320,125
1,1,2,319,125
2,1,3,318,125
3,1,4,317,125
4,1,5,316,125
5,1,6,315,125
6,1,7,314,125
7,1,8,313,125
8,1,9,312,125
9,1,10,311,125


## 4. Fit the six operating regimes

This mirrors DS-A/Paper 1. The regime label itself is **not** the DS-B shift variable; DS-B measures distance from the central operating envelope **within each regime**.

In [5]:
setting_scaler = StandardScaler()

settings_scaled = setting_scaler.fit_transform(
    train_raw[OPERATING_SETTINGS]
)

kmeans = KMeans(
    n_clusters=N_REGIMES,
    random_state=SEED,
    n_init=10
)

train_raw["regime"] = kmeans.fit_predict(
    settings_scaled
)

regime_summary = (
    train_raw
    .groupby("regime")
    .agg(
        observations=("unit_id", "size"),
        engines=("unit_id", "nunique"),
        mean_RUL=("RUL", "mean"),
        mean_setting_1=("setting_1", "mean"),
        mean_setting_2=("setting_2", "mean"),
        mean_setting_3=("setting_3", "mean")
    )
    .reset_index()
)

regime_summary["observation_pct"] = (
    100 * regime_summary["observations"] / len(train_raw)
)

display(regime_summary.round(3))

,regime,observations,engines,mean_RUL,mean_setting_1,mean_setting_2,mean_setting_3,observation_pct
0,0,15395,249,92.759,42.003,0.840,100.0,25.135
1,1,9224,249,93.098,10.003,0.251,100.0,15.060
2,2,9139,249,93.273,25.003,0.621,60.0,14.921
3,3,9091,249,93.099,20.003,0.701,100.0,14.843
4,4,9238,249,92.744,0.002,0.000,100.0,15.083
5,5,9162,249,93.095,35.003,0.841,100.0,14.959


## 5. Split engines before defining the operating envelope

In [6]:
all_engines = np.array(
    sorted(train_raw["unit_id"].unique())
)

train_engines, temp_engines = train_test_split(
    all_engines,
    test_size=0.30,
    random_state=SEED
)

cal_engines, test_engines = train_test_split(
    temp_engines,
    test_size=0.50,
    random_state=SEED
)

print("Train engines:", len(train_engines))
print("Calibration engines:", len(cal_engines))
print("Test engines:", len(test_engines))

assert set(train_engines).isdisjoint(set(cal_engines))
assert set(train_engines).isdisjoint(set(test_engines))
assert set(cal_engines).isdisjoint(set(test_engines))

print("✓ Engine-level isolation verified.")

Train engines: 174
Calibration engines: 37
Test engines: 38
✓ Engine-level isolation verified.


## 6. Estimate the central operating envelope from training engines only

For each regime, each operating setting is standardized using the mean and standard deviation observed on **training engines only**.

The envelope score is:

\[
d = \sqrt{rac{z_1^2 + z_2^2 + z_3^2}{3}}
\]

A row is considered **inside** the central envelope when its score is at or below the training-engine 80th percentile for its regime. It is **outside** when above that threshold.

This keeps all six regimes in scope while testing edge-of-envelope conditions.

In [7]:
envelope_fit = train_raw[
    train_raw["unit_id"].isin(train_engines)
].copy()

regime_stats = (
    envelope_fit
    .groupby("regime")[OPERATING_SETTINGS]
    .agg(["mean", "std"])
)

def get_regime_location_scale(stats, regime):
    means = np.array([
        stats.loc[regime, (s, "mean")]
        for s in OPERATING_SETTINGS
    ], dtype=float)

    stds = np.array([
        stats.loc[regime, (s, "std")]
        for s in OPERATING_SETTINGS
    ], dtype=float)

    # Protect against nearly constant operating-setting dimensions.
    stds = np.where(
        np.isfinite(stds) & (stds > 1e-8),
        stds,
        1.0
    )

    return means, stds


def add_envelope_score(df, stats):
    out = df.copy()
    score = np.full(len(out), np.nan, dtype=float)

    for regime in sorted(out["regime"].unique()):
        idx = out.index[out["regime"] == regime]
        means, stds = get_regime_location_scale(stats, regime)

        vals = out.loc[idx, OPERATING_SETTINGS].to_numpy(dtype=float)
        z = (vals - means) / stds

        score[out.index.get_indexer(idx)] = np.sqrt(
            np.mean(z ** 2, axis=1)
        )

    out["envelope_score"] = score
    return out


train_scored = add_envelope_score(
    envelope_fit.reset_index(drop=True),
    regime_stats
)

regime_thresholds = (
    train_scored
    .groupby("regime")["envelope_score"]
    .quantile(ENVELOPE_QUANTILE)
    .to_dict()
)

print("Per-regime envelope thresholds:")
for regime in sorted(regime_thresholds):
    print(
        f"Regime {regime}: "
        f"{regime_thresholds[regime]:.4f}"
    )

Per-regime envelope thresholds:
Regime 0: 1.0048
Regime 1: 1.0066
Regime 2: 1.0095
Regime 3: 1.0127
Regime 4: 1.0183
Regime 5: 1.0088


## 7. Score the full trajectories and label inside/outside envelope

In [8]:
trajectory_df = add_envelope_score(
    train_raw.reset_index(drop=True),
    regime_stats
)

trajectory_df["envelope_threshold"] = (
    trajectory_df["regime"].map(regime_thresholds)
)

trajectory_df["inside_envelope"] = (
    trajectory_df["envelope_score"]
    <= trajectory_df["envelope_threshold"]
)

trajectory_df = trajectory_df.sort_values(
    ["unit_id", "time_cycles"]
).reset_index(drop=True)

envelope_summary = (
    trajectory_df
    .groupby(["regime", "inside_envelope"])
    .agg(
        observations=("unit_id", "size"),
        engines=("unit_id", "nunique"),
        mean_score=("envelope_score", "mean")
    )
    .reset_index()
)

display(envelope_summary.round(3))

,regime,inside_envelope,observations,engines,mean_score
0,0,False,3070,249,1.193
1,0,True,12325,249,0.652
2,1,False,1867,249,1.193
3,1,True,7357,249,0.655
4,2,False,1830,249,1.191
5,2,True,7309,249,0.656
6,3,False,1810,249,1.198
7,3,True,7281,249,0.653
8,4,False,1861,249,1.200
9,4,True,7377,249,0.652


## 8. Verify DS-B endpoint availability before modeling

The test group should contain enough **outside-envelope** endpoint observations across multiple regimes and engines. If one regime has very few observations, keep it in the analysis but report the small sample size rather than retuning the threshold after seeing results.

In [9]:
availability_rows = []

for split_name, engines in [
    ("Train engines", train_engines),
    ("Calibration engines", cal_engines),
    ("Test engines", test_engines)
]:
    part = trajectory_df[
        trajectory_df["unit_id"].isin(engines)
    ]

    for regime in sorted(part["regime"].unique()):
        p = part[part["regime"] == regime]

        availability_rows.append({
            "Split": split_name,
            "Regime": int(regime),
            "Inside": int(p["inside_envelope"].sum()),
            "Outside": int((~p["inside_envelope"]).sum()),
            "Engines": int(p["unit_id"].nunique())
        })

availability = pd.DataFrame(availability_rows)
display(availability)

test_outside_rows = trajectory_df[
    trajectory_df["unit_id"].isin(test_engines)
    & (~trajectory_df["inside_envelope"])
]

print("Outside-envelope test rows:", len(test_outside_rows))
print(
    "Test engines with outside-envelope observations:",
    test_outside_rows["unit_id"].nunique()
)
print(
    "Regimes represented outside envelope:",
    sorted(test_outside_rows["regime"].unique())
)

,Split,Regime,Inside,Outside,Engines
0,Train engines,0,8612,2152,174
1,Train engines,1,5127,1282,174
2,Train engines,2,5104,1274,174
3,Train engines,3,5083,1271,174
4,Train engines,4,5193,1298,174
5,Train engines,5,5150,1288,174
6,Calibration engines,0,1824,455,37
7,Calibration engines,1,1108,277,37
8,Calibration engines,2,1123,278,37
9,Calibration engines,3,1066,273,37


Outside-envelope test rows: 1865
Test engines with outside-envelope observations: 38
Regimes represented outside envelope: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5)]


## 9. Fit sensor scaling using training-engine, inside-envelope observations only

This prevents outside-envelope test conditions from influencing sensor normalization.

In [10]:
sensor_scaler = StandardScaler()

scaler_fit_df = trajectory_df[
    trajectory_df["unit_id"].isin(train_engines)
    & trajectory_df["inside_envelope"]
].copy()

sensor_scaler.fit(
    scaler_fit_df[SELECTED_SENSORS]
)

trajectory_scaled = trajectory_df.copy()

trajectory_scaled[SELECTED_SENSORS] = sensor_scaler.transform(
    trajectory_scaled[SELECTED_SENSORS]
)

print("Sensor scaler fitted.")
print("Rows used:", len(scaler_fit_df))
print("Engines used:", scaler_fit_df["unit_id"].nunique())

Sensor scaler fitted.
Rows used: 34269
Engines used: 174


## 10. Leakage-safe 60-cycle window construction

Windows always come from the original consecutive engine trajectory.

Eligibility is determined by the **prediction endpoint**:
- train/calibration: endpoint must be inside the central envelope;
- test: endpoint must be outside the central envelope.

This is the primary DS-B design. Historical cycles remain physically consecutive and may include mixed operating states, which is appropriate for a temporal prognostics model whose current prediction is being evaluated at an edge/outside-envelope operating state.

In [11]:
def create_ds_b_windows(
    df,
    engine_ids,
    endpoint_mode,
    window_size=60
):
    X_sensors = []
    X_regimes = []
    y = []
    ids = []
    cycles = []
    endpoint_regimes = []
    endpoint_scores = []
    endpoint_thresholds = []

    for engine_id in sorted(engine_ids):

        eng = (
            df[df["unit_id"] == engine_id]
            .sort_values("time_cycles")
            .reset_index(drop=True)
        )

        if len(eng) < window_size:
            continue

        for end_idx in range(window_size - 1, len(eng)):

            start_idx = end_idx - window_size + 1
            window = eng.iloc[start_idx:end_idx + 1]
            endpoint = eng.iloc[end_idx]

            is_inside = bool(endpoint["inside_envelope"])

            if endpoint_mode == "inside":
                if not is_inside:
                    continue

            elif endpoint_mode == "outside":
                if is_inside:
                    continue

            else:
                raise ValueError(
                    "endpoint_mode must be 'inside' or 'outside'"
                )

            cycle_values = window["time_cycles"].to_numpy()

            if not np.all(np.diff(cycle_values) == 1):
                raise ValueError(
                    f"Non-consecutive window detected for engine {engine_id}"
                )

            X_sensors.append(
                window[SELECTED_SENSORS].to_numpy(
                    dtype=np.float32
                )
            )

            X_regimes.append(
                window["regime"].to_numpy(
                    dtype=np.int32
                )
            )

            y.append(float(endpoint["RUL"]))
            ids.append(int(engine_id))
            cycles.append(int(endpoint["time_cycles"]))
            endpoint_regimes.append(int(endpoint["regime"]))
            endpoint_scores.append(float(endpoint["envelope_score"]))
            endpoint_thresholds.append(float(endpoint["envelope_threshold"]))

    return (
        np.asarray(X_sensors, dtype=np.float32),
        np.asarray(X_regimes, dtype=np.int32),
        np.asarray(y, dtype=np.float32),
        np.asarray(ids, dtype=np.int32),
        np.asarray(cycles, dtype=np.int32),
        np.asarray(endpoint_regimes, dtype=np.int32),
        np.asarray(endpoint_scores, dtype=np.float32),
        np.asarray(endpoint_thresholds, dtype=np.float32)
    )

## 11. Construct train, calibration and DS-B test windows

In [12]:
(
    X_train_sensors,
    reg_train,
    y_train_ds,
    ids_train,
    cycles_train,
    reg_train_endpoint,
    score_train_endpoint,
    threshold_train_endpoint
) = create_ds_b_windows(
    trajectory_scaled,
    train_engines,
    endpoint_mode="inside",
    window_size=WINDOW_SIZE
)

(
    X_cal_sensors,
    reg_cal,
    y_cal_ds,
    ids_cal,
    cycles_cal,
    reg_cal_endpoint,
    score_cal_endpoint,
    threshold_cal_endpoint
) = create_ds_b_windows(
    trajectory_scaled,
    cal_engines,
    endpoint_mode="inside",
    window_size=WINDOW_SIZE
)

(
    X_test_sensors,
    reg_test,
    y_test_ds,
    ids_test,
    cycles_test,
    reg_test_endpoint,
    score_test_endpoint,
    threshold_test_endpoint
) = create_ds_b_windows(
    trajectory_scaled,
    test_engines,
    endpoint_mode="outside",
    window_size=WINDOW_SIZE
)

print("TRAIN:", X_train_sensors.shape, y_train_ds.shape)
print("CAL  :", X_cal_sensors.shape, y_cal_ds.shape)
print("TEST :", X_test_sensors.shape, y_test_ds.shape)

TRAIN: (26030, 60, 14) (26030,)
CAL  : (5510, 60, 14) (5510,)
TEST : (1408, 60, 14) (1408,)


## 12. Hard validation of the DS-B split

In [13]:
assert set(ids_train).isdisjoint(set(ids_cal))
assert set(ids_train).isdisjoint(set(ids_test))
assert set(ids_cal).isdisjoint(set(ids_test))

assert np.all(
    score_train_endpoint <= threshold_train_endpoint + 1e-7
)

assert np.all(
    score_cal_endpoint <= threshold_cal_endpoint + 1e-7
)

assert np.all(
    score_test_endpoint > threshold_test_endpoint - 1e-7
)

assert X_train_sensors.shape[1] == WINDOW_SIZE
assert X_cal_sensors.shape[1] == WINDOW_SIZE
assert X_test_sensors.shape[1] == WINDOW_SIZE

print("✓ No engine leakage.")
print("✓ Train/calibration endpoints are inside envelope.")
print("✓ Test endpoints are outside envelope.")
print("✓ Every sequence contains 60 consecutive cycles.")
print(
    "Test regimes represented:",
    sorted(np.unique(reg_test_endpoint))
)
print(
    "Test engines represented:",
    len(np.unique(ids_test))
)

✓ No engine leakage.
✓ Train/calibration endpoints are inside envelope.
✓ Test endpoints are outside envelope.
✓ Every sequence contains 60 consecutive cycles.
Test regimes represented: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5)]
Test engines represented: 38


## 13. Add six-channel regime one-hot representation

In [14]:
import tensorflow as tf

reg_train_oh = tf.keras.utils.to_categorical(
    reg_train,
    num_classes=N_REGIMES
)

reg_cal_oh = tf.keras.utils.to_categorical(
    reg_cal,
    num_classes=N_REGIMES
)

reg_test_oh = tf.keras.utils.to_categorical(
    reg_test,
    num_classes=N_REGIMES
)

X_train_ds = np.concatenate(
    [X_train_sensors, reg_train_oh],
    axis=-1
)

X_cal_ds = np.concatenate(
    [X_cal_sensors, reg_cal_oh],
    axis=-1
)

X_test_ds = np.concatenate(
    [X_test_sensors, reg_test_oh],
    axis=-1
)

print("Final model inputs:")
print("Train:", X_train_ds.shape)
print("Cal  :", X_cal_ds.shape)
print("Test :", X_test_ds.shape)

assert X_train_ds.shape[-1] == 20

print("✓ 14 sensor channels + 6 regime channels = 20 features.")

Final model inputs:
Train: (26030, 60, 20)
Cal  : (5510, 60, 20)
Test : (1408, 60, 20)
✓ 14 sensor channels + 6 regime channels = 20 features.


## P2-E8 point-model training protocol for Shift-V2

The P2-E8 architecture/hyperparameters are frozen, while weights are retrained using only the DS-B-allowed training distribution. The calibration-engine split is used solely as validation for early stopping in this point-model gate.

Do not reuse this validation split for later conformal calibration; UQ-V1 needs an untouched calibration protocol.


In [15]:
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense, LSTM
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

EPOCHS = 60
BATCH_SIZE = 64
PATIENCE = 10

def set_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def build_p2e8_point_model():
    inp = Input(shape=(WINDOW_SIZE, 20), name="p2e8_input")
    x = Dense(18, use_bias=False, name="linear_projection_18d")(inp)
    x = LSTM(64, name="lstm_core")(x)
    out = Dense(1, name="rul_head")(x)

    model = Model(inp, out, name="P2_E8_DS_B_60cycle")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model

set_all_seeds(SEED)
p2e8_model = build_p2e8_point_model()
p2e8_model.summary()


Model: "P2_E8_DS_B_60cycle"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ p2e8_input (InputLayer)         │ (None, 60, 20)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ linear_projection_18d (Dense)   │ (None, 60, 18)         │           360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_core (LSTM)                │ (None, 64)             │        21,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rul_head (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,673 (84.66 KB)

 Trainable params: 21,673 (84.66 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
set_all_seeds(SEED)

DS_B_CKPT = os.path.join(OUTPUT_DIR, "P2_E8_DS_B_best.keras")

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        DS_B_CKPT,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=0
    )
]

history = p2e8_model.fit(
    X_train_ds,
    y_train_ds,
    validation_data=(X_cal_ds, y_cal_ds),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
    shuffle=True
)

print("Training stopped after:", len(history.history["loss"]), "epochs")
print("Best checkpoint:", DS_B_CKPT)


Epoch 1/60
407/407 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 5588.4648 - mae: 64.2966 - val_loss: 4098.7646 - val_mae: 54.9820
Epoch 2/60
407/407 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 3209.2661 - mae: 49.0598 - val_loss: 2610.1365 - val_mae: 45.1170
Epoch 3/60
407/407 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 2225.1323 - mae: 42.0590 - val_loss: 2016.1144 - val_mae: 40.5695
Epoch 4/60
407/407 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 1873.3580 - mae: 39.0243 - val_loss: 1834.9038 - val_mae: 38.7275
Epoch 5/60
407/407 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 1781.0015 - mae: 37.8812 - val_loss: 1796.1218 - val_mae: 38.0756
Epoch 6/60
407/407 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 1764.7086 - mae: 37.5047 - val_loss: 1790.5112 - val_mae: 37.8676
Epoch 7/60
407/407 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 1744.3860 - mae: 37.2297 - val_loss: 1656.5201 - val_mae: 36.6835
Epoch 8/60
407/407 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 936.0540 - mae: 26.4203 - val_loss: 582.4758 - val_mae: 

## Shift-V2 / DS-B point-model evaluation and export

The outside-envelope test endpoints remain untouched until this final evaluation.


In [17]:
pred_test = p2e8_model.predict(
    X_test_ds,
    batch_size=512,
    verbose=0
).reshape(-1)

test_rmse = float(np.sqrt(mean_squared_error(y_test_ds, pred_test)))
test_mae = float(mean_absolute_error(y_test_ds, pred_test))
test_bias = float(np.mean(pred_test - y_test_ds))

print("=" * 60)
print("P2-E8 SHIFT-V2 / DS-B POINT PERFORMANCE")
print("=" * 60)
print(f"RMSE : {test_rmse:.3f}")
print(f"MAE  : {test_mae:.3f}")
print(f"Bias : {test_bias:.3f}")
print(f"N    : {len(y_test_ds)}")
print(f"Paper 1 DS-B point-RMSE reference: 23.999")
print(f"RMSE change vs Paper 1 reference: {test_rmse - 23.999:+.3f}")

life_stage = np.where(
    y_test_ds > 80, "Early",
    np.where(y_test_ds > 40, "Mid", "Late")
)

stage_rows = []
for stage in ["Early", "Mid", "Late"]:
    mask = life_stage == stage
    if not np.any(mask):
        continue
    stage_rows.append({
        "Life_Stage": stage,
        "N": int(mask.sum()),
        "RMSE": float(np.sqrt(mean_squared_error(y_test_ds[mask], pred_test[mask]))),
        "MAE": float(mean_absolute_error(y_test_ds[mask], pred_test[mask])),
        "Bias_pred_minus_true": float(np.mean(pred_test[mask] - y_test_ds[mask]))
    })

ds_b_stage_metrics = pd.DataFrame(stage_rows)
display(ds_b_stage_metrics)


P2-E8 SHIFT-V2 / DS-B POINT PERFORMANCE
RMSE : 17.130
MAE  : 12.226
Bias : -0.540
N    : 1408
Paper 1 DS-B point-RMSE reference: 23.999
RMSE change vs Paper 1 reference: -6.869


,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,Early,791,16.185452,11.430183,-6.210918
1,Mid,315,23.694468,18.851559,8.914387
2,Late,302,9.815756,7.399129,4.452783


In [18]:
ds_b_export = pd.DataFrame({
    "unit_id": np.asarray(ids_test).astype(int),
    "cycle": np.asarray(cycles_test).astype(int),
    "RUL": np.asarray(y_test_ds, dtype=float),
    "prediction": np.asarray(pred_test, dtype=float),
    "regime": np.asarray(reg_test_endpoint).astype(int),
    "envelope_score": np.asarray(score_test_endpoint, dtype=float),
    "envelope_threshold": np.asarray(threshold_test_endpoint, dtype=float)
})

DS_B_EXPORT = os.path.join(
    OUTPUT_DIR,
    "DS_B_P2E8_predictions.csv"
)

DS_B_STAGE_EXPORT = os.path.join(
    OUTPUT_DIR,
    "DS_B_P2E8_stage_metrics.csv"
)

ds_b_export.to_csv(DS_B_EXPORT, index=False)
ds_b_stage_metrics.to_csv(DS_B_STAGE_EXPORT, index=False)

print("Saved prediction export:", DS_B_EXPORT)
print("Saved stage metrics:", DS_B_STAGE_EXPORT)
print("Rows:", len(ds_b_export))
display(ds_b_export.head())

required = {
    "unit_id", "cycle", "RUL", "prediction", "regime",
    "envelope_score", "envelope_threshold"
}
assert required.issubset(ds_b_export.columns)
assert len(ds_b_export) == len(y_test_ds)
print("PASS: DS-B export is ready for the main Paper 2 Shift-V3 notebook.")


Saved prediction export: /content/drive/MyDrive/FD004_Paper2_Results/DS_B_P2E8_predictions.csv
Saved stage metrics: /content/drive/MyDrive/FD004_Paper2_Results/DS_B_P2E8_stage_metrics.csv
Rows: 1408


,unit_id,cycle,RUL,prediction,regime,envelope_score,envelope_threshold
0,17,70,81.0,122.979836,1,1.371685,1.006563
1,17,71,80.0,121.545837,4,1.091058,1.018319
2,17,73,78.0,122.239159,2,1.354804,1.009493
3,17,78,73.0,122.117088,1,1.126035,1.006563
4,17,81,70.0,119.906311,2,1.092483,1.009493


PASS: DS-B export is ready for the main Paper 2 Shift-V3 notebook.
